# Interne en externe links
Dit notebook ondersteunt het verkennen, analyseren van klik functionaliteit binnen AskDelphi.

### Connectie met AskDelphi opzetten

In [45]:
import pprint
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.import_digicoach import Import

import_service = Import()
topic_tools = TopicTools(import_service.client, import_service.project)

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


### Overzicht aanwezige Topics

In [46]:
# Alle topics
topics = topic_tools.fetch_topiclist()
# pprint.pprint(topics)

# Alle unieke topic types
unique_types = {t["topicTypeName"] for t in topics}
pprint.pprint(unique_types)

{'Afbeelding',
 'Bestand',
 'ConnectPeople',
 'Digitale Coach Homepagina',
 'Digitale Coach Procespagina',
 'Externe link',
 'Favorieten',
 'Hiërarchie',
 'Homepagina',
 'Inhoud / Artikel',
 'Interactieve afbeelding',
 'Intranet',
 'Menu',
 'Proces',
 'Stap',
 'Taak',
 'Uitklapbare sectie',
 'Verzameling onderwerpen',
 'Voorgedefinieerde zoekopdracht',
 'X - Pagina Structuur Voorgedefinieerde Zoekopdracht'}


### Overzicht bronnen taaksjabloon

In [47]:
from ask_delphi_api.convert_taaksjabloon import read_dir, convert_doc_to_json
import pprint
import json
from ask_delphi_api.import_digicoach import Import

paths = read_dir('/Users/baasa03/Digicoach/taaksjabloon_demo')
json_coaches = []

for path in paths:
    try:
        print("\n\n")
        json_coach = convert_doc_to_json(path)
        json_coaches.append(json_coach)
    except ValueError as e:
        print(e)

json_digicoach = json.loads(json_coaches[0])

LIC - Taaksjablonen Sanering v1.0.docx



retrieved 9 tables from doc /Users/baasa03/Digicoach/taaksjabloon_demo/LIC - Taaksjablonen Sanering v1.0.docx
Found title: Sanering (LIC)
Found tag: Directie CAP
Found tag: Keten Inning en betalingsverkeer
Found step: Open postbak in WAB
Found step: Selecteer alleen kwijtscheldings / saneringsverzoeken
Found step: In behandeling nemen poststuk saneringen
Found step: Controleer taken op je naam
Found task: Selecteer Zaak
Found step: Check op vermogensbestanddelen
Found step: Check op mogelijkheid aansprakelijk stellen
Found step: Check op teruggaven en goeder trouw
Found step: Check op ambtshalve aanslagen
Found step: Vastleggen acties
Found task: Quick scan verzoek
Found step: Controleer competentie
Found step: Controleer de datum van binnenkomst
Found step: Check op benodigde stukken
Found step: Check op onbehandelde zaken
Found step: Controleer volledigheid verzoek
Found step: Vastleggen brief of actie in INL
Found step: Acties bij verzoek is

In [48]:
# Bronnenlijst taaksjabloon
sources = json_digicoach["sources"]

for source in sources:
    pprint.pprint(source)

{'link': 'https://www.belastingdienst.nl/wps/wcm/connect/bldcontentnl/themaoverstijgend/programmas_en_formulieren/kwijtschelding_van_belasting_enof_premie_voor_ondernemingen',
 'titel': 'Kwijtscheldingsformulier Ondernemers',
 'type': 'Bronnen'}
{'link': 'https://connectpeople.belastingdienst.nl/files/form/anonymous/api/library/1fa8a980-ab6d-4c9a-a059-da1fea2c3dbf/document/1a372d7d-d07f-448a-8f2b-efa5c9690ec8/media/6980%20LIC%20-%20applicatie%20overzicht.pdf',
 'titel': 'Applicatie overzicht',
 'type': 'Applicaties'}
{'link': 'Word document (zelfde link als in de DC Sanering (Taskforce)',
 'titel': 'GDC - Standaardzinnen voor de sanering',
 'type': 'Handleidingen en instructies'}
{'link': 'Word document (zelfde link als in de DC Sanering (Taskforce)',
 'titel': 'LIC - memo - tijdelijke instructie coronasaneringen',
 'type': 'Handleidingen en instructies'}
{'link': 'Word document (zelfde link als in de DC Sanering (Taskforce)',
 'titel': 'LIC - checklist - bij tijdelijke instructie sane

### TODO overnemen in code

In [ ]:
import re
from typing import List, Dict

def get_topic_by_title(title: str):
    topic = next((t for t in topics if t["title"] == title), None)
    return topic

def create_link(description: str, target_topic_id: str) -> str:
    link = f"\xa0<doppio-link " \
        f"target=\"{target_topic_id}\" " \
        f"use=\"default\" " \
        f"view=\"default\" " \
        f"title=\"{description}\" " \
        f"thumbnail=\"\" " \
        f"link=\"tenant/{import_service.client.tenant_id}/project/{import_service.client.project_id}/acl/{import_service.client.acl_entry_id}/topic/{target_topic_id}/edit\">" \
        f"{description}</doppio-link><span>\xa0</span>"
    return link

def hyperlink_html(description: str, sources: List[Dict]) -> str:
    """
    Maakt één regex die alle titels bevat, gesorteerd van lang naar kort.
    Hierdoor wint bij overlappende matches de langste term per positie.
    Losse kortere woorden elders in de tekst worden wél gelinkt.
    """
    # Filter alleen titels die een topic hebben en maak mapping title -> topic
    title_to_topic = {}
    titles = []
    for src in sources:
        title = src["titel"]
        topic = get_topic_by_title(title)
        if topic is not None:
            titles.append(title)
            title_to_topic[title.lower()] = topic  # voor case-insensitive lookup

    if not titles:
        return description

    # Sorteer titels van lang naar kort en bouw één alternatiepatroon
    titles_sorted = sorted(titles, key=len, reverse=True)
    pattern = re.compile("|".join(re.escape(t) for t in titles_sorted), re.IGNORECASE)

    def _repl(m: re.Match) -> str:
        matched_text = m.group(0)
        # Zoek de canonical title (case-insensitive)
        canonical = matched_text.lower()
        # Vind de juiste title key (op basis van lengte-sortering kan exact zoeken misgaan bij casing)
        # Daarom: kies de eerste title die case-insensitive gelijk is aan de match.
        for t in titles_sorted:
            if t.lower() == canonical:
                topic = title_to_topic[canonical]
                return create_link(matched_text, topic["topicGuid"])
        # fallback: geen link
        return matched_text

    return pattern.sub(_repl, description)

### Opvragen topic op basis titel

In [71]:
for source in sources:
    topic = get_topic_by_title(source["titel"])
    # print(f"{topic["topicGuid"]}, {topic["title"]}, {topic["topicTypeName"]}, {source["link"]}")

### Test creatie Externe link (Intranet)

In [72]:
link = create_link("KRB Klantbeeld", "2e04df19-54fa-4341-be5f-0d9c5d1d1635")
pprint.pprint(link)

('\xa0<doppio-link target="2e04df19-54fa-4341-be5f-0d9c5d1d1635" use="default" '
 'view="default" title="KRB Klantbeeld" thumbnail="" '
 'link="tenant/0be6d42b-c278-44e6-888e-ba122840d690/project/397296f6-20dd-45cd-8459-250db2725140/acl/4ecd88f2-979b-4fb0-a95d-175d499bc375/topic/2e04df19-54fa-4341-be5f-0d9c5d1d1635/edit">KRB '
 'Klantbeeld</doppio-link><span>\xa0</span>')


### Test zoek bronverwijzingen en vervang deze door een hyperlink

In [76]:
text = "Deze tekst gaat over Kwijtscheldingsformulier ondernemers in 2026."
content = hyperlink_html(text, sources)
pprint.pprint(content)

('Deze tekst gaat over \xa0<doppio-link '
 'target="1f7e44ad-c58f-4150-915a-827cdb81dbff" use="default" view="default" '
 'title="Kwijtscheldingsformulier ondernemers" thumbnail="" '
 'link="tenant/0be6d42b-c278-44e6-888e-ba122840d690/project/397296f6-20dd-45cd-8459-250db2725140/acl/4ecd88f2-979b-4fb0-a95d-175d499bc375/topic/1f7e44ad-c58f-4150-915a-827cdb81dbff/edit">Kwijtscheldingsformulier '
 'ondernemers</doppio-link><span>\xa0</span> in 2026.')


### Update content "Open postbak in WAB"

In [62]:
# import pprint

# # Haal topic-parts op.
# topicId = "1ef44c28-9974-4899-b29a-fe352c9128d6"
# content = topic_tools.get_topic_parts(topicId=topicId)
# body_part = ""

# # Selecteer part uit topic met daarin de content.
# body_part = None
# groups = content['topicEditorData']['groups']
# for group in groups:
#     for part in group['parts']:
#         if part["partId"] == "body":
#             body_part = part['editors'][0]['value']['richTextEditor']['value']

# pprint.pp(body_part)

In [ ]:
topic_id = "04c5a67f-7b79-4df3-a3e5-ad734c8b18ce"
topic_version_id = topic_tools.get_topicVersionId(topic_id)

import_service.add_content_to_topic(topic_id, topic_version_id, content)

topic_tools.checkin(topic_id)

import_service.publiceer(topic_id)

2026-02-23T19:59:28Z
